# 🔋 Battery State-of-Health (SoH) Estimation — Week 3

**WIDSS — Windowed Intelligent Drive-cycle State eStimation**

This notebook covers:
1. Synthetic EV drive-cycle simulation using the `widss` package
2. Exploratory Data Analysis (EDA)
3. Feature engineering for SoC estimation
4. Baseline ML models (Linear Regression, Random Forest)
5. LSTM model training and evaluation

> **Context**: Battery SoH describes long-term degradation (capacity fade). SoC is the instantaneous charge level.  
> In this week we focus on SoC estimation as the foundation before extending to SoH.


## 0. Environment Setup

In [ ]:
# Install WIDSS (run once)
# !pip install -e ".[tensorflow]"  # from repo root

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════════
# DARK MODE STYLING (Black Background + White Text)
# ════════════════════════════════════════════════════════════════
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#1a1a1a',
    'axes.facecolor': '#0d0d0d',
    'axes.edgecolor': '#404040',
    'axes.grid': True,
    'grid.alpha': 0.2,
    'grid.color': '#505050',
    'axes.labelcolor': '#ffffff',
    'text.color': '#ffffff',
    'xtick.color': '#ffffff',
    'ytick.color': '#ffffff',
    'font.size': 10,
    'legend.fontsize': 9,
    'legend.facecolor': '#1a1a1a',
    'legend.edgecolor': '#404040'
})

print('✅ Core imports done with dark mode enabled')

In [ ]:
# WIDSS imports
import sys, os
sys.path.insert(0, os.path.abspath('../src'))  # adjust if running from repo root

from widss.simulation import BatterySimulationConfig, build_dataset
from widss.dataset import build_sequences
from widss.evaluation import rmse, mae, mape

print('✅ WIDSS package loaded')

---
## 1. Simulate EV Drive Cycle Data

In [ ]:
# Simulate 2 hours of EV driving
SEED = 42
DURATION_S = 7200  # 2 hours

cfg = BatterySimulationConfig(
    capacity_ah=60.0,
    soc_init=0.95,
    dt_s=1.0,
    internal_resistance_ohm=0.02,
    ocv_min_v=3.0,
    ocv_max_v=4.2
)

df = build_dataset(duration_s=DURATION_S, config=cfg, seed=SEED)
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'SoC range     : [{df.soc.min():.3f}, {df.soc.max():.3f}]')
df.head(10)

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Statistical summary ──────────────────────────────────────────────────────
df.describe().round(4)

In [ ]:
# ── Time-series overview ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

t = df['time_s'] / 3600  # hours

axes[0].plot(t, df['current_a'], color='#00CCFF', lw=1.2, label='Current (A)')
axes[0].axhline(0, color='#808080', lw=0.8, ls='--')
axes[0].set_ylabel('Current (A)', color='white', fontsize=11)
axes[0].set_title('EV Drive Cycle — Simulated Battery Data', fontsize=12, fontweight='bold', color='white')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].tick_params(colors='white')

axes[1].plot(t, df['voltage_v'], color='#FF9500', lw=1.2, label='Terminal Voltage (V)')
axes[1].set_ylabel('Voltage (V)', color='white', fontsize=11)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].tick_params(colors='white')

axes[2].plot(t, df['soc'], color='#00FF41', lw=1.5, label='SoC')
axes[2].set_xlabel('Time (hours)', color='white', fontsize=11)
axes[2].set_ylabel('SoC', color='white', fontsize=11)
axes[2].set_ylim(0, 1.05)
axes[2].legend(loc='upper right', fontsize=10)
axes[2].tick_params(colors='white')

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors_dist = ['#00CCFF', '#FF9500', '#00FF41']
cols = ['current_a', 'voltage_v', 'soc']
titles = ['Current Distribution', 'Voltage Distribution', 'SoC Distribution']

for ax, col, color, title in zip(axes, cols, colors_dist, titles):
    ax.hist(df[col], bins=60, color=color, alpha=0.7, edgecolor='#ffffff', linewidth=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold', color='white')
    ax.set_xlabel(col, color='white', fontsize=10)
    ax.set_ylabel('Count', color='white', fontsize=10)
    ax.tick_params(colors='white')

plt.suptitle('Feature Distributions', fontsize=13, fontweight='bold', color='white', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation matrix ───────────────────────────────────────────────────────
import seaborn as sns

corr = df[['current_a', 'voltage_v', 'soc']].corr()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'},
            annot_kws={'color': 'white', 'fontsize': 11})
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold', color='white')
for text in ax.texts:
    text.set_color('white')
plt.tight_layout()
plt.show()

In [ ]:
# ── OCV–SoC relationship (key physics relationship) ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(df['soc'], df['voltage_v'], c=df['current_a'],
                cmap='RdBu_r', alpha=0.4, s=5, edgecolors='none')
cbar = plt.colorbar(sc, label='Current (A) — blue=charging, red=discharging')
cbar.ax.tick_params(colors='white')
cbar.set_label('Current (A) — blue=charging, red=discharging', color='white', fontsize=10)
ax.set_xlabel('State of Charge', color='white', fontsize=11)
ax.set_ylabel('Terminal Voltage (V)', color='white', fontsize=11)
ax.set_title('OCV–SoC Relationship (colored by current)', fontsize=12, fontweight='bold', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

In [ ]:
# Add derived features useful for SoC estimation
df_feat = df.copy()

# Rolling statistics (1-minute window = 60 s at 1 Hz)
WIN = 60
df_feat['current_roll_mean'] = df_feat['current_a'].rolling(WIN, min_periods=1).mean()
df_feat['current_roll_std']  = df_feat['current_a'].rolling(WIN, min_periods=1).std().fillna(0)
df_feat['voltage_roll_mean'] = df_feat['voltage_v'].rolling(WIN, min_periods=1).mean()

# Cumulative charge (Coulomb counting) — key physics feature
df_feat['charge_ah'] = (df_feat['current_a'] * cfg.dt_s / 3600).cumsum()

# Power
df_feat['power_w'] = df_feat['current_a'] * df_feat['voltage_v']

# Lag features
for lag in [1, 5, 30]:
    df_feat[f'current_lag{lag}'] = df_feat['current_a'].shift(lag).fillna(0)
    df_feat[f'voltage_lag{lag}'] = df_feat['voltage_v'].shift(lag).fillna(method='bfill')

print(f'✅ Features after engineering: {len(df_feat.columns)} columns')
print(f'Shape: {df_feat.shape}')
print(f'New features: current_roll_mean, current_roll_std, voltage_roll_mean, charge_ah, power_w')
print(f'            + lag features (lag1, lag5, lag30)')
df_feat.head()

---
## 4. Baseline Models — Train/Test Split

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

FEATURE_COLS = [
    'current_a', 'voltage_v',
    'current_roll_mean', 'current_roll_std', 'voltage_roll_mean',
    'charge_ah', 'power_w',
    'current_lag1', 'voltage_lag1',
    'current_lag5', 'voltage_lag5',
    'current_lag30', 'voltage_lag30'
]
TARGET = 'soc'

# 80/20 train-test split (time-ordered — no shuffle!)
split_idx = int(0.8 * len(df_feat))
train = df_feat.iloc[:split_idx]
test  = df_feat.iloc[split_idx:]

X_train = train[FEATURE_COLS].values
y_train = train[TARGET].values
X_test  = test[FEATURE_COLS].values
y_test  = test[TARGET].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Data split: Train {X_train.shape} | Test {X_test.shape}')

In [ ]:
# ── Train & evaluate baseline models ─────────────────────────────────────────
results = {}

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression' : Ridge(alpha=1.0),
    'Random Forest'    : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.05,
                                                    max_depth=4, random_state=42)
}

print('Training baseline models...')
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_pred = np.clip(y_pred, 0, 1)  # SoC is bounded [0, 1]
    results[name] = {
        'RMSE' : rmse(y_test, y_pred),
        'MAE'  : mae(y_test, y_pred),
        'MAPE' : mape(y_test, y_pred),
        'pred' : y_pred
    }
    print(f'  {name:22s} RMSE={results[name]["RMSE"]:.4f}  MAE={results[name]["MAE"]:.4f}  MAPE={results[name]["MAPE"]:.2f}%')

print('\n✅ All baseline models trained')

In [ ]:
# ── Results comparison table ─────────────────────────────────────────────────
metrics_df = pd.DataFrame(
    {name: {k: v for k, v in vals.items() if k != 'pred'}
     for name, vals in results.items()}
).T.round(4)
print('\n📊 Model Performance Summary (sorted by RMSE):')
print(metrics_df.sort_values('RMSE').to_string())

In [ ]:
# ── Prediction plots ────────────────────────────────────────────────────────
t_test = test['time_s'].values / 3600

fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.flatten()
colors_models = ['#00CCFF', '#FF6B9D', '#00FF41', '#FFD700']

for ax, (name, vals), color in zip(axes, results.items(), colors_models):
    ax.plot(t_test, y_test, 'o-', color='#00FF41', lw=1.5, markersize=2, label='True SoC', alpha=0.8)
    ax.plot(t_test, vals['pred'], 's--', color=color, lw=1.2, markersize=2,
            label=f'Predicted (RMSE={vals["RMSE"]:.4f})', alpha=0.8)
    ax.set_title(name, fontsize=12, fontweight='bold', color='white')
    ax.set_xlabel('Time (hours)', color='white', fontsize=11)
    ax.set_ylabel('SoC', color='white', fontsize=11)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9, loc='best')
    ax.tick_params(colors='white')

plt.suptitle('Baseline Model Predictions vs True SoC', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

---
## 5. Feature Importance (Random Forest)

In [ ]:
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp.plot(kind='barh', ax=ax, color='#00CCFF', edgecolor='#ffffff', linewidth=0.7)
ax.set_title('Random Forest — Feature Importances', fontsize=12, fontweight='bold', color='white')
ax.set_xlabel('Importance', color='white', fontsize=11)
ax.set_ylabel('Feature', color='white', fontsize=11)
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

print('\n📌 Top 5 Most Important Features:')
print(feat_imp.tail(5).sort_values(ascending=False).to_string())

---
## 6. LSTM Sequence Model

We now use the `widss` LSTM pipeline which operates on raw sliding windows of current/voltage.

In [ ]:
from widss.model import build_lstm_soc_model, tensorflow_available

WINDOW_SIZE = 30

if not tensorflow_available():
    print('⚠️  TensorFlow not installed — skipping LSTM training.')
    print('    Run: pip install tensorflow>=2.13')
else:
    import tensorflow as tf
    print(f'✅ TensorFlow version: {tf.__version__}')

    # Build sequences
    X_seq, y_seq = build_sequences(df, window_size=WINDOW_SIZE)
    split = int(0.8 * len(X_seq))
    X_tr, X_te = X_seq[:split], X_seq[split:]
    y_tr, y_te = y_seq[:split], y_seq[split:]
    print(f'✅ Sequences built: Train {X_tr.shape} | Test {X_te.shape}')

In [ ]:
if tensorflow_available():
    model_lstm = build_lstm_soc_model(
        window_size=WINDOW_SIZE,
        feature_count=2,
        units=64,
        learning_rate=1e-3
    )
    print('LSTM Model Architecture:')
    model_lstm.summary()

In [ ]:
if tensorflow_available():
    print('Training LSTM model...')
    history = model_lstm.fit(
        X_tr, y_tr,
        validation_split=0.15,
        epochs=10,
        batch_size=64,
        verbose=1
    )
    print('\n✅ Training complete')

In [ ]:
if tensorflow_available():
    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    axes[0].plot(history.history['loss'], color='#00CCFF', lw=2, label='Train Loss')
    axes[0].plot(history.history['val_loss'], color='#FF6B9D', lw=2, label='Val Loss')
    axes[0].set_title('Loss (MSE)', fontsize=12, fontweight='bold', color='white')
    axes[0].set_xlabel('Epoch', color='white', fontsize=11)
    axes[0].set_ylabel('Loss', color='white', fontsize=11)
    axes[0].legend(loc='upper right', fontsize=10)
    axes[0].tick_params(colors='white')

    axes[1].plot(history.history.get('mae', []), color='#00CCFF', lw=2, label='Train MAE')
    axes[1].plot(history.history.get('val_mae', []), color='#FF6B9D', lw=2, label='Val MAE')
    axes[1].set_title('Mean Absolute Error', fontsize=12, fontweight='bold', color='white')
    axes[1].set_xlabel('Epoch', color='white', fontsize=11)
    axes[1].set_ylabel('MAE', color='white', fontsize=11)
    axes[1].legend(loc='upper right', fontsize=10)
    axes[1].tick_params(colors='white')

    plt.suptitle('LSTM Training Curves', fontsize=13, fontweight='bold', color='white')
    plt.tight_layout()
    plt.show()

In [ ]:
if tensorflow_available():
    # Evaluate LSTM
    y_lstm_pred = model_lstm.predict(X_te, verbose=0).flatten()
    y_lstm_pred = np.clip(y_lstm_pred, 0, 1)

    lstm_rmse = rmse(y_te, y_lstm_pred)
    lstm_mae = mae(y_te, y_lstm_pred)
    lstm_mape = mape(y_te, y_lstm_pred)

    print('\n' + '='*60)
    print('📊 LSTM RESULTS (Test Set)')
    print('='*60)
    print(f'RMSE : {lstm_rmse:.5f}')
    print(f'MAE  : {lstm_mae:.5f}')
    print(f'MAPE : {lstm_mape:.2f}%')
    print('='*60)

In [ ]:
if tensorflow_available():
    # Prediction plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    plot_steps = min(500, len(y_te))
    steps = np.arange(plot_steps)
    
    # Full trajectory
    axes[0].plot(steps, y_te[:plot_steps], 'o-', color='#00FF41', lw=1.5, markersize=3, label='True SoC', alpha=0.8)
    axes[0].plot(steps, y_lstm_pred[:plot_steps], 's--', color='#FF6B9D', lw=1.2, markersize=3,
                 label='LSTM Predicted', alpha=0.8)
    axes[0].fill_between(steps, y_te[:plot_steps] - lstm_mae, y_te[:plot_steps] + lstm_mae,
                          color='#FF6B9D', alpha=0.1, label='±MAE band')
    axes[0].set_title('LSTM SoC Prediction (first 500 test steps)', fontsize=12, fontweight='bold', color='white')
    axes[0].set_ylabel('SoC', color='white', fontsize=11)
    axes[0].legend(loc='best', fontsize=10)
    axes[0].tick_params(colors='white')
    
    # Residuals
    residuals = (y_te[:plot_steps] - y_lstm_pred[:plot_steps]) * 100
    axes[1].scatter(steps, residuals, s=15, alpha=0.6, color='#00CCFF')
    axes[1].axhline(0, color='#ffffff', ls='-', lw=0.8)
    axes[1].fill_between(steps, -lstm_mae*100, lstm_mae*100, color='#00CCFF', alpha=0.1)
    axes[1].set_xlabel('Timestep', color='white', fontsize=11)
    axes[1].set_ylabel('Error (%)', color='white', fontsize=11)
    axes[1].set_title('Residual Analysis', fontsize=12, fontweight='bold', color='white')
    axes[1].tick_params(colors='white')
    
    plt.tight_layout()
    plt.show()

---
## 7. Summary & Next Steps

In [ ]:
print('\n' + '='*70)
print('🎯 WEEK 3 SUMMARY')
print('='*70)
print(f'\nDataset Configuration:')
print(f'  • Duration        : {DURATION_S/3600:.0f} hours')
print(f'  • Battery Capacity: {cfg.capacity_ah} Ah (Li-ion)')
print(f'  • Features used   : {len(FEATURE_COLS)} engineered features')
print(f'  • Train/Test split: 80/20 (time-ordered)')

print(f'\nBaseline Model Performance (Test Set):')
for name, vals in sorted(results.items(), key=lambda x: x[1]['RMSE']):
    print(f'  • {name:22s} → RMSE: {vals["RMSE"]:.5f}, MAE: {vals["MAE"]:.5f}, MAPE: {vals["MAPE"]:.2f}%')

if tensorflow_available():
    print(f'\nLSTM Model Performance (Test Set):')
    print(f'  • RMSE: {lstm_rmse:.5f}')
    print(f'  • MAE : {lstm_mae:.5f}')
    print(f'  • MAPE: {lstm_mape:.2f}%')

print(f'\n📌 Next Steps (Week 4):')
print(f'  1. Physics-informed feature extraction (SPM model)')
print(f'  2. Transformer-based architecture for sequence modeling')
print(f'  3. SoH (capacity fade) prediction over battery lifecycle')
print(f'  4. Remaining useful life (RUL) estimation')
print('='*70 + '\n')